# Understanding Linear Regression for Signal Discovery

This notebook explains linear regression from first principles, why we use it instead of correlation, and how to interpret the results for trading signals.

## Contents
1. What is Linear Regression?
2. The Math (simplified)
3. Key Outputs: Coefficient, p-value, R², t-stat
4. Why Not Just Use Correlation?
5. Hands-On Examples with Bitcoin Data
6. Common Pitfalls

In [ ]:
# Setup
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# For plotting
try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except:
    HAS_PLOT = False
    print("matplotlib not installed - plots will be skipped")

---
## 1. What is Linear Regression?

Linear regression answers: **"If X changes, how does Y change?"**

For trading signals:
- **X** = metric value today (e.g., MVRV)
- **Y** = future return (e.g., 30-day forward return)

We're trying to find if there's a **predictable relationship** between the metric and future returns.

### The Model

```
Y = α + β·X + ε
```

Where:
- **Y** = what we're predicting (future returns)
- **X** = what we're using to predict (metric value)
- **α** (alpha) = intercept (baseline return when X=0)
- **β** (beta) = coefficient (how much Y changes per unit X)
- **ε** (epsilon) = error term (what we can't explain)

---
## 2. The Math (Simplified)

### What OLS Does

OLS = **Ordinary Least Squares**

It finds the line that **minimizes the sum of squared errors**:

```
Minimize: Σ(actual_Y - predicted_Y)²
```

### Visual Intuition

In [ ]:
# Generate simple example data
np.random.seed(42)
n = 100

# True relationship: Y = 2 + 0.5*X + noise
X_example = np.random.uniform(0, 10, n)
Y_example = 2 + 0.5 * X_example + np.random.normal(0, 1, n)

# Fit regression
X_with_const = sm.add_constant(X_example)
model = sm.OLS(Y_example, X_with_const).fit()

print("True relationship: Y = 2 + 0.5*X + noise")
print(f"\nEstimated: Y = {model.params[0]:.2f} + {model.params[1]:.2f}*X")
print(f"\nThe OLS found the true coefficients!")

In [ ]:
if HAS_PLOT:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Left: Scatter with regression line
    ax = axes[0]
    ax.scatter(X_example, Y_example, alpha=0.5, label='Data points')
    x_line = np.linspace(0, 10, 100)
    y_line = model.params[0] + model.params[1] * x_line
    ax.plot(x_line, y_line, 'r-', linewidth=2, label='Best fit line')
    ax.set_xlabel('X (metric)')
    ax.set_ylabel('Y (return)')
    ax.set_title('Linear Regression: Finding the Best Fit Line')
    ax.legend()
    
    # Right: Residuals
    ax = axes[1]
    residuals = model.resid
    ax.scatter(X_example, residuals, alpha=0.5)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('X (metric)')
    ax.set_ylabel('Residual (error)')
    ax.set_title('Residuals: What We Cannot Explain')
    
    plt.tight_layout()
    plt.show()

---
## 3. Key Outputs

When you run a regression, you get several important statistics:

In [ ]:
# Full regression summary
print(model.summary())

### 3.1 Coefficient (β)

**What it tells you:** For each 1-unit increase in X, Y changes by β.

**For trading signals:**
- **Positive coefficient** → Higher metric value predicts higher returns → Use `above` threshold
- **Negative coefficient** → Higher metric value predicts lower returns → Use `below` threshold

In [ ]:
print(f"Coefficient: {model.params[1]:.4f}")
print(f"\nInterpretation: For each 1-unit increase in X,")
print(f"Y increases by {model.params[1]:.4f} units.")

### 3.2 Standard Error

**What it tells you:** How uncertain we are about the coefficient estimate.

- Small standard error = confident estimate
- Large standard error = uncertain estimate

In [ ]:
print(f"Coefficient: {model.params[1]:.4f}")
print(f"Standard Error: {model.bse[1]:.4f}")
print(f"\n95% Confidence Interval: {model.params[1]:.4f} ± {1.96 * model.bse[1]:.4f}")
print(f"  Lower: {model.params[1] - 1.96 * model.bse[1]:.4f}")
print(f"  Upper: {model.params[1] + 1.96 * model.bse[1]:.4f}")

### 3.3 t-statistic

**What it tells you:** How many standard errors the coefficient is from zero.

```
t-stat = coefficient / standard_error
```

**Rule of thumb:**
- |t| > 2 → Usually statistically significant
- |t| < 2 → Probably not significant

In [ ]:
t_stat = model.params[1] / model.bse[1]
print(f"t-statistic: {t_stat:.2f}")
print(f"\nThis is {abs(t_stat):.1f} standard errors away from zero.")
if abs(t_stat) > 2:
    print("→ Statistically significant!")
else:
    print("→ Not statistically significant.")

### 3.4 p-value ⭐ MOST IMPORTANT

**What it tells you:** The probability of seeing this result (or more extreme) if there's actually NO relationship.

**Interpretation:**
- p < 0.01 → Very strong evidence of relationship (***)
- p < 0.05 → Strong evidence (**)
- p < 0.10 → Weak evidence (*)
- p > 0.10 → No significant evidence

**For trading:**
- p < 0.05 means: "There's less than 5% chance this relationship is just random noise."

In [ ]:
p_value = model.pvalues[1]
print(f"p-value: {p_value:.6f}")

if p_value < 0.01:
    print("\n*** Highly significant (p < 0.01)")
    print("Less than 1% chance this is random noise.")
elif p_value < 0.05:
    print("\n** Significant (p < 0.05)")
    print("Less than 5% chance this is random noise.")
elif p_value < 0.10:
    print("\n* Marginally significant (p < 0.10)")
else:
    print("\nNot significant (p >= 0.10)")
    print("Could easily be random noise!")

### 3.5 R-squared (R²)

**What it tells you:** What percentage of the variance in Y is explained by X.

- R² = 0 → X explains nothing about Y
- R² = 1 → X perfectly explains Y

**For trading signals:**
- Don't expect high R²! Markets are noisy.
- R² = 0.02 (2%) can still be profitable
- R² = 0.05 (5%) is actually quite good for financial data

In [ ]:
print(f"R-squared: {model.rsquared:.4f} ({model.rsquared*100:.1f}%)")
print(f"\nInterpretation: X explains {model.rsquared*100:.1f}% of the variance in Y.")
print(f"The remaining {(1-model.rsquared)*100:.1f}% is unexplained (noise, other factors).")

---
## 4. Why Not Just Use Correlation?

Correlation and regression are related, but regression gives you MORE:

In [ ]:
# Correlation
correlation = np.corrcoef(X_example, Y_example)[0, 1]
print(f"Correlation: {correlation:.4f}")
print(f"\nCorrelation only tells you:")
print(f"  - Direction: {'positive' if correlation > 0 else 'negative'}")
print(f"  - Strength: {abs(correlation):.2f}")
print(f"\nBut NOT:")
print(f"  - Is it statistically significant? (need p-value)")
print(f"  - How much does Y change per unit X? (need coefficient)")
print(f"  - How confident are we? (need standard error)")

In [ ]:
# Show the relationship between correlation and R²
print(f"Correlation: {correlation:.4f}")
print(f"Correlation²: {correlation**2:.4f}")
print(f"R-squared: {model.rsquared:.4f}")
print(f"\n→ For simple regression, R² = correlation²")

### The Big Problem with Correlation Alone

Correlation doesn't tell you if the result is **statistically significant**.

Example: If you have only 10 data points, a correlation of 0.50 might not be significant.
But with 1000 data points, a correlation of 0.10 might be highly significant!

In [ ]:
# Demonstrate with different sample sizes
print("Same correlation (0.30), different sample sizes:\n")

for n in [10, 50, 100, 500, 1000]:
    # Generate data with correlation ~0.30
    np.random.seed(42)
    x = np.random.normal(0, 1, n)
    y = 0.30 * x + np.sqrt(1 - 0.30**2) * np.random.normal(0, 1, n)
    
    # Regression
    X_const = sm.add_constant(x)
    result = sm.OLS(y, X_const).fit()
    
    sig = "***" if result.pvalues[1] < 0.01 else "**" if result.pvalues[1] < 0.05 else "*" if result.pvalues[1] < 0.1 else ""
    print(f"n={n:4d}  corr={np.corrcoef(x,y)[0,1]:.2f}  p-value={result.pvalues[1]:.4f} {sig}")

**Key insight:** With more data, we can detect smaller effects with confidence!

---
## 5. Hands-On Examples with Bitcoin Data

In [ ]:
# Load Bitcoin data
DATA_DIR = Path("../data/raw")

if not DATA_DIR.exists():
    print(f"Data directory not found: {DATA_DIR}")
    print("Run 'python run.py sync' first to download data.")
else:
    # Load metrics
    metrics_to_load = ['price', 'mvrv', 'nvt', 'sopr', 'nupl']
    dfs = {}
    
    for metric in metrics_to_load:
        path = DATA_DIR / f"{metric}.parquet"
        if path.exists():
            temp = pd.read_parquet(path)
            temp = temp.set_index("time")
            temp = temp.rename(columns={"value": metric})
            dfs[metric] = temp
    
    if dfs:
        df = pd.concat(dfs.values(), axis=1).sort_index()
        
        # Add forward returns
        df['fwd_30d'] = df['price'].pct_change(30).shift(-30)
        
        print(f"Loaded {len(df)} rows")
        print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
        print(f"Columns: {list(df.columns)}")
    else:
        print("No data files found.")

In [ ]:
# Run regression for each metric
if 'df' in dir() and len(df) > 0:
    print("Regression Results: fwd_30d_return ~ metric\n")
    print(f"{'Metric':<15} {'Coef':>12} {'Std Err':>10} {'t-stat':>8} {'p-value':>10} {'R²':>8}")
    print("-" * 70)
    
    for metric in ['mvrv', 'nvt', 'sopr', 'nupl']:
        if metric not in df.columns:
            continue
        
        data = df[[metric, 'fwd_30d']].dropna()
        X = sm.add_constant(data[metric])
        y = data['fwd_30d']
        
        result = sm.OLS(y, X).fit()
        
        sig = "***" if result.pvalues[1] < 0.01 else "**" if result.pvalues[1] < 0.05 else "*" if result.pvalues[1] < 0.1 else ""
        
        print(f"{metric:<15} {result.params[1]:>+12.6f} {result.bse[1]:>10.6f} "
              f"{result.tvalues[1]:>8.2f} {result.pvalues[1]:>10.4f} {result.rsquared:>8.4f} {sig}")

### Interpreting the Results

Let's break down what each column means for your trading signals:

| Column | Meaning | Trading Implication |
|--------|---------|--------------------|
| **Coef** | How much return changes per unit metric | Sign tells you direction: negative → use "below" threshold |
| **Std Err** | Uncertainty in coefficient | Smaller = more confident |
| **t-stat** | Coefficient / Std Err | \|t\| > 2 usually significant |
| **p-value** | Probability this is noise | p < 0.05 = use this signal! |
| **R²** | Variance explained | Don't worry if small, 2-5% is good |

In [ ]:
# Detailed example with MVRV
if 'df' in dir() and 'mvrv' in df.columns:
    data = df[['mvrv', 'fwd_30d']].dropna()
    X = sm.add_constant(data['mvrv'])
    y = data['fwd_30d']
    result = sm.OLS(y, X).fit()
    
    print("=" * 60)
    print("DETAILED EXAMPLE: MVRV")
    print("=" * 60)
    print(f"\nCoefficient: {result.params[1]:+.6f}")
    print(f"  → For each 1-point increase in MVRV,")
    print(f"    30-day return changes by {result.params[1]*100:+.4f}%")
    
    print(f"\np-value: {result.pvalues[1]:.6f}")
    if result.pvalues[1] < 0.05:
        print(f"  → Statistically significant! This is a real signal.")
    else:
        print(f"  → Not significant. Could be noise.")
    
    print(f"\nR²: {result.rsquared:.4f} ({result.rsquared*100:.2f}%)")
    print(f"  → MVRV explains {result.rsquared*100:.2f}% of return variance.")
    print(f"  → This is {'good' if result.rsquared > 0.01 else 'low'} for financial data.")
    
    print(f"\nTrading implication:")
    if result.params[1] > 0 and result.pvalues[1] < 0.05:
        print(f"  → Use signal: MVRV ABOVE threshold")
    elif result.params[1] < 0 and result.pvalues[1] < 0.05:
        print(f"  → Use signal: MVRV BELOW threshold")
    else:
        print(f"  → Signal not significant enough to use.")

---
## 6. Common Pitfalls

### Pitfall 1: Confusing Correlation with Causation

In [ ]:
print("Just because MVRV predicts returns doesn't mean MVRV CAUSES returns.")
print("\nPossible explanations:")
print("  1. MVRV directly affects price (causal)")
print("  2. Both MVRV and price are driven by a third factor")
print("  3. Pure coincidence (if p-value is high)")
print("\nFor trading, we don't need causation - prediction is enough!")

### Pitfall 2: Multiple Testing Problem

If you test 100 metrics, ~5 will appear significant by chance (at p < 0.05).

In [ ]:
# Demonstrate multiple testing problem
np.random.seed(42)

print("Testing 100 RANDOM metrics (no real relationship):\n")

significant_count = 0
for i in range(100):
    # Random X, random Y - NO relationship!
    x = np.random.normal(0, 1, 500)
    y = np.random.normal(0, 1, 500)
    
    X_const = sm.add_constant(x)
    result = sm.OLS(y, X_const).fit()
    
    if result.pvalues[1] < 0.05:
        significant_count += 1

print(f"Metrics that appeared 'significant' (p < 0.05): {significant_count}")
print(f"Expected by chance: ~5")
print(f"\n⚠️  This is why we need out-of-sample validation!")

### Pitfall 3: Overfitting to a Specific Threshold

This is why we need the **smoothness check** in grid search.

In [ ]:
print("If your signal only works at threshold = 1.234567...")
print("...but fails at 1.23 and 1.24...")
print("\n→ It's probably overfit to noise!")
print("\nA robust signal works across a RANGE of thresholds.")
print("That's what 'smooth Sharpe curve' means.")

### Pitfall 4: Low R² Doesn't Mean Useless

In finance, even R² = 0.01 (1%) can be profitable!

In [ ]:
print("Why low R² can still be profitable:\n")
print("R² = 0.02 means the metric explains 2% of variance.")
print("\nBut if you:")
print("  - Trade frequently (compound small edges)")
print("  - Have good risk management")
print("  - Use multiple uncorrelated signals")
print("\n...small edges add up!")
print("\nRenaissance Technologies' Medallion Fund reportedly")
print("uses signals with R² in the 1-2% range.")

---
## Summary: What to Look For

When evaluating a potential signal:

| Check | Threshold | Why |
|-------|-----------|-----|
| **p-value** | < 0.05 | Statistically significant |
| **Coefficient sign** | Consistent | Tells you above/below |
| **Smoothness** | < 0.5 | Not overfit to one threshold |
| **Cycle consistency** | > 60% | Works across multiple periods |
| **R²** | > 0.01 | Explains SOMETHING (don't expect high) |

**Remember the analyst's rules:**
1. Use regression with p-values, not just correlation
2. Grid search: Sharpe curve must be SMOOTH
3. Must work in MULTIPLE market cycles
4. Keep to SINGLE DIGIT metrics (max 9)

In [ ]:
print("Next steps:")
print("1. Run: python -m src.miner")
print("2. Review the regression results and p-values")
print("3. Check smoothness of Sharpe curves")
print("4. Validate with walk-forward: python -m src.walk_forward")